# **Question 2: Intermediate - Code Structure & meta-programming**

In MLOps, you often need to apply cross-cutting concerns—like **logging execution time**, **retrying failed API calls**, or **validating inputs**—across dozens of different functions in your training or inference pipeline. You don't want to copy-paste this logic everywhere.

**The Scenario:**
You need to create a **Python Decorator** called `@measure_latency` that calculates how long a model prediction function takes to run.

**The Question:**
1.  How do you implement this decorator?
2.  **Critical for MLOps:** When you decorate a function, it often loses its original identity (its `__name__` and `__doc__` string). How do you prevent this loss, and why is preserving this metadata important when using frameworks like **FastAPI** or **Pickle** (often used to serialize models)?

## 🔹 1. What is Metaprogramming (Quick Intro)

Metaprogramming is a technique where **code operates on other code at runtime**.

In Python, this is very natural because:

* Functions are first-class objects
* We can pass functions as arguments
* We can modify behavior dynamically

👉 In real-world systems (especially MLOps), this helps us:

* Avoid duplication
* Add reusable behaviors (logging, retries, latency tracking)
* Keep business logic clean

---

## 🔹 2. Why Decorators (Core Idea)

Decorators are the **most practical form of metaprogramming**.

They allow us to:

> Add behavior (like latency tracking) **without modifying the original function**

---

## 🔹 3. Implementing `@measure_latency`

### ✅ Code Implementation

```python
import time
from functools import wraps

def measure_latency(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        
        result = func(*args, **kwargs)
        
        end_time = time.time()
        latency = end_time - start_time
        
        print(f"{func.__name__} executed in {latency:.4f} seconds")
        
        return result
    
    return wrapper
```

---

### ✅ Usage Example

```python
@measure_latency
def predict_model(data):
    time.sleep(1)
    return "prediction"

predict_model("sample input")
```

---

### 🔹 What Happens Internally

Python converts:

```python
@measure_latency
def predict_model():
```

Into:

```python
predict_model = measure_latency(predict_model)
```

👉 So the original function gets **wrapped with extra behavior**

---

## 🔹 4. Why This is Important in MLOps

In MLOps pipelines, we often need:

* Model inference latency tracking
* Retry logic for flaky APIs
* Logging & monitoring
* Input validation

👉 Instead of repeating logic everywhere, decorators give:

> **Reusable + clean + production-grade pipelines**

---

## 🔹 5. Problem: Metadata Loss

### ❌ What Goes Wrong

When we decorate a function:

* Original function is replaced by `wrapper`
* So we lose:

  * `__name__`
  * `__doc__`
  * annotations

Example:

```python
print(predict_model.__name__)  # wrapper (wrong)
```

---

## 🔹 6. Solution: `functools.wraps`

### ✅ Fix

```python
from functools import wraps
```

```python
def measure_latency(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper
```

---

### ✅ What `@wraps` Preserves

* `__name__`
* `__doc__`
* `__annotations__`
* `__module__`
* `__wrapped__`

---

## 🔹 7. Why Metadata is CRITICAL (MLOps + Backend)

### 🚀 In **FastAPI**

* Uses function signature & annotations for:

  * Request validation
  * Auto-generated Swagger docs

👉 Without `@wraps`:

* API docs break
* Function name shows as `wrapper`
* Debugging becomes hard

---

### 🚀 In **Pickle (Model Serialization)**

* Used to serialize models/functions
* Depends on correct function identity

👉 Without metadata:

* Deserialization can fail
* Wrong references may be stored

---

### 🚀 In Logging & Debugging

* Logs become unclear:

  ```
  wrapper executed...
  ```

  instead of:

  ```
  predict_model executed...
  ```

---

## 🔹 8. FastAPI vs Django REST Framework Insight (Senior-Level Point)

* **FastAPI** → heavily depends on function metadata (annotations, signature)
* **Django REST Framework** → less dependent (uses class-based views)

👉 But:

> In both cases, preserving metadata is a **production best practice**

---

## 🔹 9. Final Interview Answer (Short Version)

> Decorators are a form of metaprogramming that allow us to add reusable behavior like latency tracking without modifying the original function.
>
> I would implement `@measure_latency` using a wrapper function that records execution time before and after the function call.
>
> A key issue with decorators is that they replace the original function, causing loss of metadata like `__name__` and `__doc__`.
>
> To prevent this, I use `functools.wraps`, which preserves the original function’s metadata.
>
> This is especially important in MLOps systems where frameworks like FastAPI rely on function signatures for validation and documentation, and tools like Pickle rely on correct function identity for serialization.

---

## 🔥 Senior-Level Add-on (Say This → Instant Impact)

> "In production MLOps systems, I’d extend this decorator to push latency metrics to monitoring systems like Prometheus or logs instead of just printing, so we can track model performance over time."

---

## ✅ Key Takeaways (Revision)

* Decorators = clean metaprogramming tool
* Avoid duplication (cross-cutting concerns)
* Always use `@wraps`
* Critical for FastAPI, Pickle, observability
* Real-world use → monitoring, retries, logging

---

# ✅ **Metaprogramming in Python (Beyond Decorators)**

## 🔹 1. Introspection (Inspecting Code at Runtime)

👉 Ability to **look inside objects/functions/classes while the program is running**

### Examples:

* `type()`
* `dir()`
* `getattr()`
* `hasattr()`

### Example:

```python
class Model:
    def predict(self):
        pass

m = Model()

print(type(m))           # <class '__main__.Model'>
print(dir(m))            # list of attributes
print(hasattr(m, 'predict'))  # True
```

### 💡 MLOps Use Case:

* Dynamically check if a model has `predict()` or `transform()`
* Build flexible pipelines

---

## 🔹 2. Reflection (Modifying Code at Runtime)

👉 Ability to **change objects dynamically**

### Examples:

* `setattr()`
* `delattr()`

### Example:

```python
class Model:
    pass

m = Model()
setattr(m, "version", "v1")

print(m.version)  # v1
```

### 💡 MLOps Use Case:

* Add metadata to models dynamically
* Inject configs at runtime

---

## 🔹 3. Dynamic Code Execution

👉 Executing code generated at runtime

### Examples:

* `eval()`
* `exec()`

### Example:

```python
code = "x = 10 + 5"
exec(code)
print(x)  # 15
```

### ⚠️ Important:

* Dangerous if misused (security risk)

### 💡 MLOps Use Case:

* Rare, but used in:

  * dynamic rule engines
  * configurable pipelines

---

## 🔹 4. Decorators (Most Practical)

👉 Already covered — add behavior to functions/classes

### 💡 Use Cases:

* Logging
* Retry logic
* Latency tracking
* Authentication

---

## 🔹 5. Metaclasses (Advanced Level 🚀)

👉 “Class of a class” — controls how classes are created

### Example:

```python
class Meta(type):
    def __new__(cls, name, bases, dct):
        dct['version'] = "1.0"
        return super().__new__(cls, name, bases, dct)

class Model(metaclass=Meta):
    pass

print(Model.version)  # 1.0
```

### 💡 MLOps / Framework Use:

* Used internally by frameworks
* ORM systems (like Django models)
* Enforcing rules across classes

👉 **Interview Tip:**
You don’t need to use metaclasses daily, but knowing them = strong signal

---

## 🔹 6. Dynamic Class Creation

👉 Creating classes at runtime

### Example:

```python
MyClass = type("MyClass", (), {"x": 10})

obj = MyClass()
print(obj.x)  # 10
```

### 💡 Use Case:

* Plugin systems
* Dynamic schema generation

---

## 🔹 7. Monkey Patching

👉 Modifying existing code at runtime

### Example:

```python
class Model:
    def predict(self):
        return "old"

def new_predict(self):
    return "new"

Model.predict = new_predict

print(Model().predict())  # new
```

### 💡 MLOps Use Case:

* Hotfixes (not recommended in production usually)
* Testing/mocking

---

## 🔹 8. Function Annotations & Signatures

👉 Python stores metadata about function inputs

### Example:

```python
def predict(x: int) -> float:
    return float(x)

print(predict.__annotations__)
```

### 💡 MLOps Use Case:

* Used heavily by frameworks like FastAPI
* Input validation
* Auto documentation

---

# 🔥 **Interview-Level Summary (Best Answer)**

> Metaprogramming in Python includes multiple techniques beyond decorators.
> These include introspection for inspecting objects at runtime, reflection for modifying them dynamically, dynamic code execution using eval/exec, metaclasses for controlling class creation, dynamic class generation using type, and monkey patching for runtime modifications.
>
> In real-world MLOps systems, we mostly use decorators, introspection, and annotations, while metaclasses are used internally by frameworks.

---

# 🚀 **Senior-Level Insight (Say This → Strong Impact)**

> "In production systems, I prefer safer metaprogramming techniques like decorators and introspection. I avoid heavy use of exec or monkey patching because they reduce readability and can introduce debugging and security issues."

---

# 🚀 **Metaclasses in Python (Deep Dive)**

## 🔹 1. First, Understand This Hierarchy (VERY IMPORTANT)

Everything in Python follows this chain:

```
object → class → metaclass
```

👉 More concretely:

* Object → instance of a class
* Class → instance of a metaclass
* Metaclass → defines how a class is created

### Example:

```python
class MyClass:
    pass

obj = MyClass()

print(type(obj))        # MyClass
print(type(MyClass))    # type
```

👉 Key Insight:

> `type` is the **default metaclass in Python**

---

## 🔹 2. What is a Metaclass?

👉 A metaclass is:

> **A class that defines how other classes are created**

Think of it like:

* Class → blueprint of objects
* Metaclass → blueprint of classes

---

## 🔹 3. Why Do We Need Metaclasses?

Normally:

```python
class Model:
    pass
```

👉 This is simple, but what if you want to:

* Enforce rules on all classes
* Automatically add attributes
* Validate class structure

👉 That’s where metaclasses come in.

---

## 🔹 4. How Class Creation Works Internally

When Python sees:

```python
class Model:
    x = 10
```

👉 Internally it does:

```python
Model = type("Model", (), {"x": 10})
```

So:

> Classes are created using `type()`

---

## 🔹 5. Creating a Custom Metaclass

### Step 1: Define Metaclass

```python
class Meta(type):
    def __new__(cls, name, bases, dct):
        print(f"Creating class: {name}")
        return super().__new__(cls, name, bases, dct)
```

---

### Step 2: Use It

```python
class Model(metaclass=Meta):
    x = 10
```

### Output:

```
Creating class: Model
```

👉 This runs **at class creation time**, not object creation.

---

## 🔹 6. Modify Class During Creation

Let’s inject behavior 👇

```python
class Meta(type):
    def __new__(cls, name, bases, dct):
        dct["version"] = "1.0"
        return super().__new__(cls, name, bases, dct)

class Model(metaclass=Meta):
    pass

print(Model.version)  # 1.0
```

👉 You just modified the class automatically.

---

## 🔹 7. Enforcing Rules (Very Important Use Case)

Example: Force all classes to have `predict()`

```python
class Meta(type):
    def __new__(cls, name, bases, dct):
        if "predict" not in dct:
            raise TypeError("Class must implement predict()")
        return super().__new__(cls, name, bases, dct)

class GoodModel(metaclass=Meta):
    def predict(self):
        pass

class BadModel(metaclass=Meta):
    pass   # ❌ Error
```

👉 This is powerful for:

* MLOps pipelines
* Plugin systems
* Standardizing model interfaces

---

## 🔹 8. `__new__` vs `__init__` in Metaclasses

### 🔸 `__new__` (Most Important)

* Runs **before class is created**
* Used to **modify class definition**

### 🔸 `__init__`

* Runs **after class is created**
* Used for additional setup

---

## 🔹 9. Real-World Use Cases (Important for Interview)

### ✅ 1. ORMs (like Django)

* Automatically map classes → database tables

### ✅ 2. Frameworks

* Enforce structure (e.g., required methods)

### ✅ 3. MLOps Systems

* Ensure all models follow a contract:

  * must have `predict()`
  * must define metadata

### ✅ 4. Registries (Very Powerful 🔥)

Auto-register models:

```python
registry = {}

class Meta(type):
    def __new__(cls, name, bases, dct):
        new_class = super().__new__(cls, name, bases, dct)
        registry[name] = new_class
        return new_class

class ModelA(metaclass=Meta):
    pass

print(registry)
```

👉 Useful for:

* Model discovery
* Plugin architecture
* Auto-loading pipelines

---

## 🔹 10. When NOT to Use Metaclasses ❌

Be very clear on this (interview gold):

Avoid metaclasses when:

* Decorators can solve the problem
* Simpler solutions exist
* Code readability is critical

👉 Because:

> Metaclasses make code harder to understand and debug

---

## 🔥 **Interview-Ready Answer**

> A metaclass in Python is a class that defines how other classes are created. By default, Python uses the `type` metaclass.
>
> Metaclasses allow us to modify class creation, enforce rules, or automatically inject attributes and behavior.
>
> For example, in MLOps systems, we can use metaclasses to ensure all model classes implement a `predict()` method or to automatically register models in a registry.
>
> However, I prefer using metaclasses only when necessary, since they increase complexity, and simpler approaches like decorators or base classes are often sufficient.

---

## 🚀 **Senior-Level Insight (Say This → Strong Signal)**

> "In production, I’d use metaclasses mainly for enforcing contracts or building registries. For most cross-cutting concerns like logging or retries, decorators are more maintainable."

---

## ✅ Final Mental Model

| Concept   | Think Like          |
| --------- | ------------------- |
| Object    | Instance of class   |
| Class     | Blueprint of object |
| Metaclass | Blueprint of class  |

---